In [26]:
!pip install -q -U mlflow boto3 awscli dagshub

In [27]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

try:
    
    access_key = user_secrets.get_secret("AWS_ACCESS_KEY")
    secret_key = user_secrets.get_secret("AWS_SECRET_KEY")
    
    # Set environment variables
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
    
    print("✅ AWS Credentials successfully loaded!")
    print(f"Access Key ID: {access_key[:4]}****{access_key[-4:]}")
    print(f"Region: {os.environ['AWS_DEFAULT_REGION']}")

except Exception as e:
    print("❌ Failed to find secrets. Make sure you added them in Add-ons -> Secrets.") 

✅ AWS Credentials successfully loaded!
Access Key ID: AKIA****LQZB
Region: us-east-1


In [28]:
## Setup Mlflow Server

mlflow.set_tracking_uri("http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/")
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

2026/05/13 15:53:45 INFO mlflow.tracking.fluent: Experiment with name 'Exp 2 - BoW vs TfIdf' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://amzn-mlflow-bucket-26/2', creation_time=1778687625467, experiment_id='2', last_update_time=1778687625467, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={}, trace_location=None, workspace='default'>

In [29]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [30]:
df = pd.read_csv('/kaggle/working/reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [31]:
# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)]  # unigrams, bigrams, trigrams
max_features = 5000  # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")


2026/05/13 15:53:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 15:54:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run BoW_(1, 1)_RandomForest at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2/runs/a8bbad9a670b406196c1675bebc0c35b
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2


2026/05/13 15:54:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 15:54:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TF-IDF_(1, 1)_RandomForest at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2/runs/e65f3565eb574f03aef278d957d1ca4d
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2


2026/05/13 15:55:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 15:55:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2/runs/029e57e7de44468188578bd82d6215cd
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2


2026/05/13 15:56:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 15:56:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run BoW_(1, 3)_RandomForest at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2/runs/0a12a7ca3c154a89a70ebb4d4dde141c
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2


2026/05/13 15:56:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 15:56:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TF-IDF_(1, 3)_RandomForest at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2/runs/ab0329299ab84deb8b9f1ef186f84a9f
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/2
